In [2]:
# Libraries
import cv2
import numpy as np
import time

# Face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Distraction threshold (seconds)
DISTRACTION_THRESHOLD = 2.0

print("Libraries imported!")

Libraries imported!


In [3]:
# Distraction tracking variables
distraction_start = None
DISTRACTION_THRESHOLD = 2.0

# Face detector only
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Baseline face size (set when looking forward)
baseline_face_size = None

def get_distraction_label(face, frame_shape, baseline):
    """Detect distraction based on face size and position"""
    h, w = frame_shape[:2]
    fx, fy, fw, fh = face

    face_center_y  = fy + fh // 2
    frame_center_y = h // 2
    offset_y = face_center_y - frame_center_y

    # Face size ratio compared to baseline
    if baseline is not None:
        size_ratio = fw / baseline
    else:
        size_ratio = 1.0

    # If face significantly smaller than baseline = turned sideways
    if size_ratio < 0.65:
        return "LOOKING AWAY"
    elif offset_y < -100:
        return "LOOKING UP"
    elif offset_y > 100:
        return "LOOKING DOWN"
    else:
        return "FORWARD"

print("Distraction functions ready!")

Distraction functions ready!


In [1]:
import pygame
import os

pygame.mixer.init()
alarm_path    = r"C:\Users\Bushra Shahid\Desktop\IDVS\Distraction Model\alarm"
alarm_playing = False
distraction_start  = None
baseline_face_size = None
label_history      = []
HISTORY_SIZE       = 8

def play_alarm():
    """Play alarm sound"""
    global alarm_playing
    alarm_files = [f for f in os.listdir(alarm_path) if f.endswith('.mp3') or f.endswith('.wav')]
    if alarm_files and not alarm_playing:
        pygame.mixer.music.load(os.path.join(alarm_path, alarm_files[0]))
        pygame.mixer.music.play(-1)
        alarm_playing = True

def stop_alarm():
    """Stop alarm sound"""
    global alarm_playing
    if alarm_playing:
        pygame.mixer.music.stop()
        alarm_playing = False

def get_stable_label(label):
    """Smooth label using history"""
    label_history.append(label)
    if len(label_history) > HISTORY_SIZE:
        label_history.pop(0)
    return max(set(label_history), key=label_history.count)

cap = cv2.VideoCapture(0)
print("Camera started! Press Q to quit.")
print("Pehle seedha dekho - baseline set ho ga automatically")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1,
                                          minNeighbors=5, minSize=(100, 100))

    raw_label = "FORWARD"
    color     = (0, 255, 0)

    if len(faces) > 0:
        face = faces[0]
        fx, fy, fw, fh = face
        cv2.rectangle(frame, (fx, fy), (fx+fw, fy+fh), (255, 255, 0), 2)

        # Set baseline automatically when face is large (forward)
        if baseline_face_size is None or fw > baseline_face_size:
            baseline_face_size = fw

        raw_label = get_distraction_label(face, frame.shape, baseline_face_size)
        color     = (0, 255, 0) if raw_label == "FORWARD" else (0, 0, 255)

    else:
        raw_label = "FACE NOT DETECTED"
        color     = (0, 0, 255)

    # Stable label
    stable_label = get_stable_label(raw_label)

    # Distraction logic
    if stable_label == "FORWARD":
        distraction_start = None
        stop_alarm()
    else:
        if distraction_start is None:
            distraction_start = time.time()

    # Show label
    cv2.putText(frame, stable_label, (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    # Baseline info
    if baseline_face_size:
        cv2.putText(frame, f"Baseline: {baseline_face_size}px",
                    (10, frame.shape[0]-20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

    # Alert and alarm
    if distraction_start is not None:
        elapsed = time.time() - distraction_start
        if elapsed >= DISTRACTION_THRESHOLD:
            play_alarm()
            cv2.putText(frame, f"DISTRACTION ALERT! {elapsed:.1f}s",
                        (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow('Distraction Detection', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
stop_alarm()
print("Camera stopped!")

pygame 2.6.1 (SDL 2.28.4, Python 3.11.0)
Hello from the pygame community. https://www.pygame.org/contribute.html


NameError: name 'cv2' is not defined